# Fetch CAL FIRE Perimeters

Downloads historic CAL FIRE perimeters for LA County from the official FRAP ArcGIS endpoints, normalises the `FIRE_NAME` field, filters to LA County, and saves to `data/processed/fire_perimeters.geojson`.

In [1]:
from pathlib import Path

import geopandas as gpd
import requests

## Configuration

Set `LOCAL_PERIMETERS` to a file path to skip the download and load from a local GeoJSON/shapefile instead.  
Set `OUT_PATH` to override the default save location.

In [ ]:
LOCAL_PERIMETERS = None  # e.g. "path/to/fires.geojson", or None to download

ROOT = Path("../..")
OUT_PATH = ROOT / "data" / "processed" / "fire_perimeters.geojson"

CRS = "EPSG:3310"  # CA Albers

YEAR_MIN = 2020
YEAR_MAX = 2025

CALFIRE_ENDPOINTS = [
    "https://egis.fire.ca.gov/arcgis/rest/services/FRAP/FirePerimeters_FS/FeatureServer/0/query",
    "https://services1.arcgis.com/jUJYIo9tSA7EHvfZ/arcgis/rest/services/"
    "California_Historic_Fire_Perimeters/FeatureServer/0/query",
]

CALFIRE_PARAMS = {
    "where": f"YEAR_ >= {YEAR_MIN} AND YEAR_ <= {YEAR_MAX}",
    "outFields": "*",
    "f": "geojson",
    "resultRecordCount": "1000",
}

## Download or Load Perimeters

In [ ]:
if LOCAL_PERIMETERS:
    fires = gpd.read_file(LOCAL_PERIMETERS)
    print(f"Perimeters loaded from file: {len(fires)} records")
else:
    print(f"Downloading CAL FIRE perimeters ({YEAR_MIN}–{YEAR_MAX})...")
    fires = None
    last_exc = None
    for url in CALFIRE_ENDPOINTS:
        try:
            pages = []
            offset = 0
            while True:
                params = {**CALFIRE_PARAMS, "resultOffset": str(offset)}
                resp = requests.get(url, params=params, timeout=60)
                resp.raise_for_status()
                data = resp.json()
                features = data.get("features", [])
                if not features:
                    break
                pages.extend(features)
                if len(features) < int(CALFIRE_PARAMS["resultRecordCount"]):
                    break  # last page
                offset += len(features)

            if not pages:
                raise ValueError("Empty response")

            fires = gpd.GeoDataFrame.from_features(pages, crs="EPSG:4326")
            print(f"Downloaded from {url.split('/arcgis')[0]}: {len(fires)} records")
            break
        except Exception as exc:
            print(f"  Endpoint failed ({url.split('/arcgis')[0]}): {exc}")
            last_exc = exc

    if fires is None:
        raise RuntimeError(
            f"All CAL FIRE endpoints failed — {last_exc}\n"
            "Download the shapefile manually from:\n"
            "  https://frap.fire.ca.gov/frap-projects/fire-perimeters/\n"
            "Then set LOCAL_PERIMETERS above and re-run."
        )

## Normalise and Filter

In [4]:
fires = fires.to_crs(CRS)

name_col = next((c for c in fires.columns if "FIRE_NAME" in c.upper()), None)
fires["FIRE_NAME"] = fires[name_col].str.strip().str.upper() if name_col else "UNKNOWN"

county_col = next((c for c in fires.columns if c.upper() in ("COUNTY", "UNIT_ID")), None)
if county_col and fires[county_col].str.upper().str.contains("LOS ANGELES|LRA").any():
    fires = fires[fires[county_col].str.upper().str.contains("LOS ANGELES", na=False)]
    print(f"Filtered to LA County: {len(fires)} records")

fires[["FIRE_NAME", "geometry"]].head()

,FIRE_NAME,geometry
0,PALISADES,"MULTIPOLYGON (((131998.056 -435368.234, 131998..."
1,EATON,"MULTIPOLYGON (((176706.656 -418166.172, 176705..."
2,HUGHES,"MULTIPOLYGON (((132176.399 -380697.134, 132154..."
3,KENNETH,"POLYGON ((121966.746 -426575.295, 121943.238 -..."
4,HURST,"POLYGON ((140773.831 -408331.971, 140774.237 -..."


## Save

In [5]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fires.to_file(OUT_PATH, driver="GeoJSON")
print(f"Saved: {OUT_PATH}  ({len(fires)} records)")

Saved: ../../data/processed/fire_perimeters.geojson  (1000 records)
